<img src="Pictures/MLimg3.jpg">
<hr>
<img src="Pictures/MLimg4.jpg">
<hr>

What is bagging and Boosting in data science? <br>
 - Bagging gives equal weight to each model, whereas in Boosting technique, the new models are weighted based on their results.

# Bagging

<img src="Pictures/MLimg5.jpg">
Various models are built parallel on various samples and then the various models <b> vote </b> to give the final model and hence prediction.

The best example for bagging is the "random forest" model which takes different random samples and apply DecisionTree for 'n' time. Then, the result is evaluated by the help of voting.

#### In bagging, a random sample of data in a training set is selected with replacement—meaning that the individual data points can be chosen more than once. However, in pasting, a sample is chosen once in the execution.

# Boosting

<a href="https://www.youtube.com/watch?v=R5FB1ZUejXM"> Look at the difference among CatBoost, XGBoost and LightGBM </a>

In [11]:
import pandas as pd

turbo = pd.read_csv("../ML_Datasets/turboaz.csv")
needy_columns = ["Buraxilish ili","Yurush","Qiymet"]
data = turbo[needy_columns].copy()      # Extracted columns
data['Yurush'] = data['Yurush'].str.replace("km","").str.replace(" ","")
data['Yurush'] = data['Yurush'].astype('int64')

currency = data["Qiymet"].apply(lambda x: x.split()[1])
data["currency"] = currency

def exchange_to_AZN(val, curr):
    if curr == "$":
        return int(val)*1.7
    return int(val)

data["Qiymet"] = data["Qiymet"].apply(lambda x: exchange_to_AZN(x.split()[0], x.split()[1]))
data.drop(["currency"], axis = 1, inplace = True)
regression_data = data.copy()
regression_data.head(3)

,Buraxilish ili,Yurush,Qiymet
0,1999,366000,12500.0
1,2014,102000,53550.0
2,2002,469700,11700.0


In [12]:
from sklearn.model_selection import train_test_split
Xr = regression_data.iloc[:,:-1]
yr = regression_data.iloc[:,-1]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(Xr, yr, test_size=0.1, random_state=42)

In [13]:
class_data = pd.read_csv("../ML_Datasets/UCI_Credit_Card.csv")
class_data.drop(["ID"], axis=1, inplace=True)
class_data.head(2)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,2,2,2,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1


In [18]:
Xc = class_data.iloc[:, 0:23]
yc = class_data.iloc[:, -1]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(Xc, yc, test_size=0.1, random_state=42)

### LightGBM

In [19]:
!pip install lightgbm

In [20]:
from lightgbm import LGBMRegressor
model = LGBMRegressor()
model.fit(X_train_r, y_train_r)
model.score(X_test_r, y_test_r)

0.8587564983313112

In [21]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
model = LGBMClassifier()
model.fit(X_train_c, y_train_c)
y_pred = model.predict(X_test_c)
accuracy_score(y_pred, y_test_c)

0.8216666666666667

### XGBoost

In [22]:
!pip install xgboost

In [23]:
import xgboost
print(xgboost.__version__)

1.6.1


In [24]:
from xgboost import XGBClassifier
help(XGBClassifier)

Help on class XGBClassifier in module xgboost.sklearn:

class XGBClassifier(XGBModel, sklearn.base.ClassifierMixin)
 |  XGBClassifier(*, objective: Union[str, Callable[[numpy.ndarray, numpy.ndarray], Tuple[numpy.ndarray, numpy.ndarray]], NoneType] = 'binary:logistic', use_label_encoder: bool = False, **kwargs: Any) -> None
 |  
 |  Implementation of the scikit-learn API for XGBoost classification.
 |  
 |  
 |  Parameters
 |  ----------
 |  
 |      n_estimators : int
 |          Number of boosting rounds.
 |  
 |      max_depth :  Optional[int]
 |          Maximum tree depth for base learners.
 |      max_leaves :
 |          Maximum number of leaves; 0 indicates no limit.
 |      max_bin :
 |          If using histogram-based algorithm, maximum number of bins per feature
 |      grow_policy :
 |          Tree growing policy. 0: favor splitting at nodes closest to the node, i.e. grow
 |          depth-wise. 1: favor splitting at nodes with highest loss change.
 |      learning_rate : 

In [25]:
xgbt = XGBClassifier(max_depth = 2,
             learning_rate = 0.2,
             objective  = "multi:softmax",
             num_class = 2,
             booster = "gbtree",
             n_estimarors = 10,
             random_state = 123)

xgbt.fit(X_train_c, y_train_c)
xgbt_pred = xgbt.predict(X_test_c)
accuracy_score(xgbt_pred, y_test_c)

[00:47:21] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.6.0/src/learner.cc:627: 
Parameters: { "n_estimarors" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.




0.8203333333333334

In [26]:
from xgboost import XGBRegressor
model = XGBRegressor()
model.fit(X_train_r, y_train_r)
model.score(X_test_r, y_test_r)

0.9237046556412684

### CatBoost

In [28]:
!pip install catboost

In [29]:
from catboost import CatBoostRegressor
model = CatBoostRegressor(iterations=2,
                          learning_rate=1,
                          depth=2)
model.fit(X_train_r, y_train_r)
model.score(X_test_r, y_test_r)

0:	learn: 5800.7106697	total: 113ms	remaining: 113ms
1:	learn: 4081.1339042	total: 114ms	remaining: 0us


0.8891598466914072